In [1]:
import numpy as np

from pyfstat.utils import get_sft_as_arrays

In [4]:
d1 = get_sft_as_arrays("./data/no_glitch/100-100Hz/simCW0/H-3840_H1_1800SFT_simCW0_NBF0099Hz0W0002Hz0-1368970000-6912000.sft")

25-12-01 19:37:52.131 pyfstat.utils.sft INFO    : Loading 3840 SFTs from H1...


In [5]:
d2 = get_sft_as_arrays("/home/hoitim.cheung/SFTs/o4a_data/SFTs/narrowBand_age300yr/240days/H1/100/H-7338_H1_1800SFT_O4RUN+R1+CGDSCALIBSTRAINCLEANGATEDG02+WTKEY5_NBF0099Hz0W0003Hz0-1368980712-20472783.sft")

25-12-01 19:37:52.360 pyfstat.utils.sft INFO    : Loading 7338 SFTs from H1...


In [8]:
d1[1]['H1']

array([1368970000, 1368971800, 1368973600, ..., 1375876600, 1375878400,
       1375880200], shape=(3840,))

In [12]:
np.diff(d1[1]['H1'])

array([1800, 1800, 1800, ..., 1800, 1800, 1800], shape=(3839,))

In [17]:
(np.diff(d2[1]['H1']) > 1800 ).sum()

np.int64(599)

In [18]:
(np.diff(d1[1]['H1']) > 1800 ).sum()

np.int64(0)

In [19]:
import logging
import os
from typing import Dict, Optional, Tuple

import lal
import lalpulsar
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

logger = logging.getLogger(__name__)


def get_sft_as_arrays(
    sftfilepattern: str,
    fMin: Optional[float] = None,
    fMax: Optional[float] = None,
    constraints: Optional[lalpulsar.SFTConstraints] = None,
) -> Tuple[np.ndarray, Dict, Dict]:
    """
    Read binary SFT files into NumPy arrays.

    Parameters
    ----------
    sftfilepattern:
        Pattern to match SFTs using wildcards (`*?`) and ranges [0-9];
        multiple patterns can be given separated by colons.
    fMin, fMax:
        Restrict frequency range to `[fMin, fMax]`.
        If None, retrieve the full frequency range.
    constraints:
        Constrains to be fed into XLALSFTdataFind to specify detector,
        GPS time range or timestamps to be retrieved.

    Returns
    ----------
    freqs: np.ndarray
        The frequency bins in each SFT. These will be the same for each SFT,
        so only a single 1D array is returned.
    times: Dict
        The SFT start times as a dictionary of 1D arrays, one for each detector.
        Keys correspond to the official detector names as returned by
        lalpulsar.ListIFOsInCatalog.
    data: Dict
        A dictionary of 2D arrays of the complex Fourier amplitudes of the SFT data
        for each detector in each frequency bin at each timestamp.
        Keys correspond to the official detector names as returned by
        lalpulsar.ListIFOsInCatalog.
    """

    constraints = constraints or lalpulsar.SFTConstraints()
    if fMin is None and fMax is None:
        fMin = fMax = -1
    elif fMin is None or fMax is None:
        raise ValueError("Need either none or both of fMin, fMax.")

    sft_catalog = lalpulsar.SFTdataFind(sftfilepattern, constraints)
    ifo_labels = lalpulsar.ListIFOsInCatalog(sft_catalog)

    logger.info(
        f"Loading {sft_catalog.length} SFTs from {', '.join(ifo_labels.data)}..."
    )
    multi_sfts = lalpulsar.LoadMultiSFTs(sft_catalog, fMin, fMax)
    logger.debug("done!")

    times = {}
    amplitudes = {}

    old_frequencies = None
    for ind, ifo in enumerate(ifo_labels.data):
        sfts = multi_sfts.data[ind]

        times[ifo] = np.array([sft.epoch.gpsSeconds for sft in sfts.data])
        amplitudes[ifo] = np.array([sft.data.data for sft in sfts.data]).T

        nbins, nsfts = amplitudes[ifo].shape

        logger.debug(f"{nsfts} retrieved from {ifo}.")

        f0 = sfts.data[0].f0
        df = sfts.data[0].deltaF
        frequencies = np.linspace(f0, f0 + (nbins - 1) * df, nbins)

        if (old_frequencies is not None) and not np.allclose(
            frequencies, old_frequencies
        ):
            raise ValueError(
                f"Frequencies don't match between {ifo_labels.data[ind - 1]} and {ifo}"
            )
        old_frequencies = frequencies

    return frequencies, times, amplitudes
